In [1]:
import numpy as np
import pandas as pd
import re

import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk

In [2]:
nltk.download("stopwords")
nltk.download("punkt")

[nltk_data] Downloading package stopwords to C:\Users\Arpita
[nltk_data]     Jain\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\Arpita
[nltk_data]     Jain\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
df = pd.read_csv("../dataset/Resume/Resume.csv")

In [4]:
df.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [5]:
df.shape

(2484, 4)

In [6]:
df.columns
# Resume_str to category yhi ml model hoga hmara 


Index(['ID', 'Resume_str', 'Resume_html', 'Category'], dtype='str')

In [7]:
df.isnull().sum()

ID             0
Resume_str     0
Resume_html    0
Category       0
dtype: int64

In [8]:
df.duplicated().sum()

0

In [9]:
df["Category"].value_counts()

Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
FINANCE                   118
ENGINEERING               118
ACCOUNTANT                118
FITNESS                   117
AVIATION                  117
SALES                     116
HEALTHCARE                115
CONSULTANT                115
BANKING                   115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64

In [10]:
print(df["Resume_str"][0])

         HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System (ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inventory control, loss preve

In [11]:
resume = df["Resume_str"][0]

print(resume[:1500])

         HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System (ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inventory control, loss preve

In [12]:
resume = df["Resume_str"][0]

display(resume)

"         HR ADMINISTRATOR/MARKETING ASSOCIATE\n\nHR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System (ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inventory control, loss pr

In [13]:
from IPython.display import Markdown, display

resume = df["Resume_str"][0]

display(Markdown(f"```\n{resume}\n```"))

```
         HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System (ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inventory control, loss prevention, safety, time management, leadership and performance assessment.        Experience      HR Administrator/Marketing Associate

HR Administrator     Dec 2013   to   Current      Company Name   －   City  ,   State     Helps to develop policies, directs and coordinates activities such as employment, compensation, labor relations, benefits, training, and employee services.  Prepares employee separation notices and related documentation  Keeps records of benefits plans participation such as insurance and pension plan, personnel transactions such as hires, promotions, transfers, performance reviews, and terminations, and employee statistics for government reporting.  Advises management in appropriate resolution of employee relations issues.  Administers benefits programs such as life, health, dental, insurance, pension plans, vacation, sick leave, leave of absence, and employee assistance.     Marketing Associate     Designed and created marketing collateral for sales meetings, trade shows and company executives.  Managed the in-house advertising program consisting of print and media collateral pieces.  Assisted in the complete design and launch of the company's website in 2 months.  Created an official company page on Facebook to facilitate interaction with customers.  Analyzed ratings and programming features of competitors to evaluate the effectiveness of marketing strategies.         Advanced Medical Claims Analyst     Mar 2012   to   Dec 2013      Company Name   －   City  ,   State     Reviewed medical bills for the accuracy of the treatments, tests, and hospital stays prior to sanctioning the claims.  Trained to interpret the codes (ICD-9, CPT) and terminology commonly used in medical billing to fully understand the paperwork that is submitted by healthcare providers.  Required to have organizational and analytical skills as well as computer skills, knowledge of medical terminology and procedures, statistics, billing standards, data analysis and laws regarding medical billing.         Assistant General Manager     Jun 2010   to   Dec 2010      Company Name   －   City  ,   State     Performed duties including but not limited to, budgeting and financial management, accounting, human resources, payroll and purchasing.  Established and maintained close working relationships with all departments of the hotel to ensure maximum operation, productivity, morale and guest service.  Handled daily operations and reported directly to the corporate office.  Hired and trained staff on overall objectives and goals with an emphasis on high customer service.  Marketing and Advertising, working on public relations with the media, government and local businesses and Chamber of Commerce.         Executive Support / Marketing Assistant     Jul 2007   to   Jun 2010      Company Name   －   City  ,   State     Provided assistance to various department heads - Executive, Marketing, Customer Service, Human Resources.  Managed front-end operations to ensure friendly and efficient transactions.  Ensured the swift resolution of customer issues to preserve customer loyalty while complying with company policies.  Exemplified the second-to-none customer service delivery in all interactions with customers and potential clients.         Reservation & Front Office Manager     Jun 2004   to   Jul 2007      Company Name   －   City  ,   State          Owner/ Partner     Dec 2001   to   May 2004      Company Name   －   City  ,   State          Price Integrity Coordinator     Aug 1999   to   Dec 2001      Company Name   －   City  ,   State          Education      N/A  ,   Business Administration   1999     Jefferson College   －   City  ,   State       Business Administration  Marketing / Advertising         High School Diploma  ,   College Prep. studies   1998     Sainte Genevieve Senior High   －   City  ,   State       Awarded American Shrubel Leadership Scholarship to Jefferson College         Skills     Accounting, ads, advertising, analytical skills, benefits, billing, budgeting, clients, Customer Service, data analysis, delivery, documentation, employee relations, financial management, government relations, Human Resources, insurance, labor relations, layout, Marketing, marketing collateral, medical billing, medical terminology, office, organizational, payroll, performance reviews, personnel, policies, posters, presentations, public relations, purchasing, reporting, statistics, website.    
```

In [14]:
resume = df["Resume_str"][0]

for i, line in enumerate(resume.split("."), 1):
    print(f"{i}. {line.strip()}\n")

1. HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management

2. Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service

3. Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System (ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inventory control, loss prevention, saf

In [15]:
from textwrap import fill

resume = df["Resume_str"][0]

print(fill(resume, width=100))

         HR ADMINISTRATOR/MARKETING ASSOCIATE  HR ADMINISTRATOR       Summary     Dedicated Customer
Service Manager with 15+ years of experience in Hospitality and Customer Service Management.
Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic
commitment to customer service.         Highlights         Focused on customer satisfaction  Team
management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled
multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor
Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton
Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality
systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System
(ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inventory
control, loss prevent

In [16]:
df["Clean_Resume"] = df["Resume_str"]

In [17]:
df["Clean_Resume"] = df["Resume_str"]

In [18]:
import os
import sys

sys.path.append(os.path.abspath(".."))

In [19]:
from utils.text_cleaner import clean_text

In [20]:
sample = df["Clean_Resume"][0]

print(clean_text(sample[:500]))

         hr administrator marketing associate

hr administrator       summary     dedicated customer service manager with     years of experience in hospitality and customer service management    respected builder and leader of customer focused teams  strives to instill a shared  enthusiastic commitment to customer service          highlights         focused on customer satisfaction  team management  marketing savvy  conflict resolution techniques     training and development  skilled multi task


In [21]:
df["Clean_Resume"] = df["Clean_Resume"].apply(clean_text)

In [22]:
print(df["Clean_Resume"][0][:500])

         hr administrator marketing associate

hr administrator       summary     dedicated customer service manager with     years of experience in hospitality and customer service management    respected builder and leader of customer focused teams  strives to instill a shared  enthusiastic commitment to customer service          highlights         focused on customer satisfaction  team management  marketing savvy  conflict resolution techniques     training and development  skilled multi task


In [23]:
sample = """
Visit https://google.com
My Linkedin is www.linkedin.com/in/arpita
"""

print(clean_text(sample))


visit  
my linkedin is  



In [24]:
df["Clean_Resume"] = df["Clean_Resume"].apply(clean_text)

In [25]:
print(df["Clean_Resume"][0][:500])

         hr administrator marketing associate

hr administrator       summary     dedicated customer service manager with     years of experience in hospitality and customer service management    respected builder and leader of customer focused teams  strives to instill a shared  enthusiastic commitment to customer service          highlights         focused on customer satisfaction  team management  marketing savvy  conflict resolution techniques     training and development  skilled multi task


In [26]:
sample = """
Name : Arpita Jain
Email : arpita@gmail.com
"""

print(clean_text(sample))


name   arpita jain
email    



In [27]:
from utils.text_cleaner import clean_text

In [28]:
sample = """
Name : Arpita Jain
Email : arpita@gmail.com
"""

print(clean_text(sample))


name   arpita jain
email    



In [29]:
import re

text = "Email : arpita@gmail.com"

print(re.sub(r'\S+@\S+', ' ', text))

Email :  


In [30]:
sample = """
Name : Arpita
Phone : +91 9876543210
Mobile : 9876543210
"""

print(clean_text(sample))


name   arpita
phone    
mobile    



In [31]:
text = re.sub(r'\+?\d[\d\s()-]{8,}\d', ' ', text)

In [32]:
sample = """
Name : Arpita
Phone : +91 9876543210
Mobile : 9876543210
"""

print(clean_text(sample))


name   arpita
phone    
mobile    



In [33]:
import importlib
import utils.text_cleaner

importlib.reload(utils.text_cleaner)

from utils.text_cleaner import clean_text

In [34]:
sample = """
Name : Arpita
Phone : +91 9876543210
Mobile : 9876543210
"""

print(clean_text(sample))


name   arpita
phone    
mobile    



In [35]:
import re

text = "Phone : +91 9876543210"

print(re.sub(r'\+?\d[\d\s()-]{8,}\d', ' ', text))

Phone :  


In [36]:
text = "Mobile : 9876543210"

print(re.sub(r'\+?\d[\d\s()-]{8,}\d', ' ', text))

Mobile :  


In [37]:
sample = """
Python, SQL (2025)
C++
Machine-Learning!!!
"""

print(clean_text(sample))


python  sql       
c  
machine learning   



In [38]:
import importlib
import utils.text_cleaner

importlib.reload(utils.text_cleaner)

from utils.text_cleaner import clean_text

In [39]:
sample = """
Python, SQL (2025)
C++
Machine-Learning!!!
"""

print(clean_text(sample))


python  sql       
c  
machine learning   



In [40]:
from nltk.tokenize import word_tokenize

sample = df["Clean_Resume"][0]

tokens = word_tokenize(sample)

print(tokens[:50])

['hr', 'administrator', 'marketing', 'associate', 'hr', 'administrator', 'summary', 'dedicated', 'customer', 'service', 'manager', 'with', 'years', 'of', 'experience', 'in', 'hospitality', 'and', 'customer', 'service', 'management', 'respected', 'builder', 'and', 'leader', 'of', 'customer', 'focused', 'teams', 'strives', 'to', 'instill', 'a', 'shared', 'enthusiastic', 'commitment', 'to', 'customer', 'service', 'highlights', 'focused', 'on', 'customer', 'satisfaction', 'team', 'management', 'marketing', 'savvy', 'conflict', 'resolution']


In [41]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to C:\Users\Arpita
[nltk_data]     Jain\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Arpita
[nltk_data]     Jain\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Arpita
[nltk_data]     Jain\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [42]:
from nltk.tokenize import word_tokenize

sample = df["Clean_Resume"][0]

tokens = word_tokenize(sample)

print(tokens[:20])

['hr', 'administrator', 'marketing', 'associate', 'hr', 'administrator', 'summary', 'dedicated', 'customer', 'service', 'manager', 'with', 'years', 'of', 'experience', 'in', 'hospitality', 'and', 'customer', 'service']


In [43]:
import nltk

print(nltk.__version__)

3.9.1


In [44]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

print(list(stop_words)[:20])

['mustn', 'am', 'for', 'couldn', 'shan', 'wouldn', 'very', 'yours', 'over', 'won', 'my', "don't", 'if', 'll', "he'd", "it'd", 'in', 'below', 'and', 'he']


In [45]:
filtered_words = [
    word
    for word in tokens
    if word.lower() not in stop_words
]

print(filtered_words[:50])

['hr', 'administrator', 'marketing', 'associate', 'hr', 'administrator', 'summary', 'dedicated', 'customer', 'service', 'manager', 'years', 'experience', 'hospitality', 'customer', 'service', 'management', 'respected', 'builder', 'leader', 'customer', 'focused', 'teams', 'strives', 'instill', 'shared', 'enthusiastic', 'commitment', 'customer', 'service', 'highlights', 'focused', 'customer', 'satisfaction', 'team', 'management', 'marketing', 'savvy', 'conflict', 'resolution', 'techniques', 'training', 'development', 'skilled', 'multi', 'tasker', 'client', 'relations', 'specialist', 'accomplishments']


In [46]:
df["Tokens"] = df["Clean_Resume"].apply(word_tokenize)

df["Filtered_Words"] = df["Tokens"].apply(
    lambda words: [
        word for word in words
        if word.lower() not in stop_words
    ]
)

In [47]:
df[["Tokens", "Filtered_Words"]].head()

,Tokens,Filtered_Words
0,"[hr, administrator, marketing, associate, hr, ...","[hr, administrator, marketing, associate, hr, ..."
1,"[hr, specialist, us, hr, operations, summary, ...","[hr, specialist, us, hr, operations, summary, ..."
2,"[hr, director, summary, over, years, experienc...","[hr, director, summary, years, experience, rec..."
3,"[hr, specialist, summary, dedicated, driven, a...","[hr, specialist, summary, dedicated, driven, d..."
4,"[hr, manager, skill, highlights, hr, skills, h...","[hr, manager, skill, highlights, hr, skills, h..."


In [48]:
import nltk

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to C:\Users\Arpita
[nltk_data]     Jain\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Arpita
[nltk_data]     Jain\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [49]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

In [50]:
words = ["working", "worked", "works", "studies", "running"]

lemmatized = [lemmatizer.lemmatize(word) for word in words]

print(lemmatized)

['working', 'worked', 'work', 'study', 'running']


In [51]:
words = ["working", "worked", "works", "studies", "running"]

lemmatized = [
    lemmatizer.lemmatize(word, pos="v")
    for word in words
]

print(lemmatized)

['work', 'work', 'work', 'study', 'run']


In [52]:
df["Lemmatized_Words"] = df["Filtered_Words"].apply(
    lambda words: [
        lemmatizer.lemmatize(word, pos="v")
        for word in words
    ]
)

In [53]:
print(df["Lemmatized_Words"][0][:50])

['hr', 'administrator', 'market', 'associate', 'hr', 'administrator', 'summary', 'dedicate', 'customer', 'service', 'manager', 'years', 'experience', 'hospitality', 'customer', 'service', 'management', 'respect', 'builder', 'leader', 'customer', 'focus', 'team', 'strive', 'instill', 'share', 'enthusiastic', 'commitment', 'customer', 'service', 'highlight', 'focus', 'customer', 'satisfaction', 'team', 'management', 'market', 'savvy', 'conflict', 'resolution', 'techniques', 'train', 'development', 'skilled', 'multi', 'tasker', 'client', 'relations', 'specialist', 'accomplishments']


In [54]:
from utils.skill_extractor import extract_skills

In [55]:
sample = """
Python
SQL
Machine Learning
Power BI
Pandas
NumPy
"""

print(extract_skills(sample))

['python', 'sql', 'power bi', 'numpy', 'pandas', 'machine learning']


In [56]:
df["Detected_Skills"] = df["Clean_Resume"].apply(extract_skills)

In [57]:
df[["Category","Detected_Skills"]].head()

,Category,Detected_Skills
0,HR,[aws]
1,HR,[git]
2,HR,[excel]
3,HR,[excel]
4,HR,"[excel, aws]"


In [58]:
import spacy

print(spacy.__version__)

3.8.14


In [59]:
import spacy

nlp = spacy.load("en_core_web_sm")

In [60]:
text = """
I know Python, SQL, Machine Learning and Power BI.
"""

doc = nlp(text)

for token in doc:
    print(token.text)



I
know
Python
,
SQL
,
Machine
Learning
and
Power
BI
.




In [61]:
for token in doc:
    print(token.text, "---->", token.lemma_)


 ----> 

I ----> I
know ----> know
Python ----> Python
, ----> ,
SQL ----> SQL
, ----> ,
Machine ----> Machine
Learning ----> Learning
and ----> and
Power ----> Power
BI ----> BI
. ----> .

 ----> 



In [62]:
text = """
I know Python, SQL, Machine Learning and Power BI.
"""

doc = nlp(text)

for token in doc:
    print(token.text)
for token in doc:
    print(token.text, "---->", token.lemma_)



I
know
Python
,
SQL
,
Machine
Learning
and
Power
BI
.



 ----> 

I ----> I
know ----> know
Python ----> Python
, ----> ,
SQL ----> SQL
, ----> ,
Machine ----> Machine
Learning ----> Learning
and ----> and
Power ----> Power
BI ----> BI
. ----> .

 ----> 



In [63]:
for token in doc:
    print(token.text, token.pos_)


 SPACE
I PRON
know VERB
Python PROPN
, PUNCT
SQL PROPN
, PUNCT
Machine PROPN
Learning PROPN
and CCONJ
Power PROPN
BI PROPN
. PUNCT

 SPACE


In [64]:
text = """
Arpita Jain worked at Microsoft in Delhi in 2025.
"""

doc = nlp(text)

for ent in doc.ents:
    print(ent.text, "------>", ent.label_)

Arpita Jain ------> PERSON
Microsoft ------> ORG
Delhi ------> GPE
2025 ------> DATE


In [65]:
resume = df["Resume_str"][0]

doc = nlp(resume)

for ent in doc.ents:
    print(ent.text, ent.label_)

15+ years DATE
Hospitality GPE
Team ORG
Missouri GPE
DOT ORG
IHG ORG
Customer Loyalty and Marketing ORG
Hilton GPE
Training Certification PERSON
Micros NORP
Fidelio     PERSON
Dec 2013 DATE
Current      Company Name ORG
State     Helps ORG
Prepares PERSON
Keeps ORG
Assisted PERSON
2 months DATE
2012 DATE
Dec 2013 DATE
CPT ORG
2010 DATE
Dec 2010 DATE
Established PERSON
daily DATE
Marketing and Advertising ORG
Chamber of Commerce ORG
2007 DATE
Jun 2010      Company Name   －    PERSON
- Executive ORG
Marketing, Customer Service ORG
second ORDINAL
Reservation & Front Office ORG
2004 DATE
2007 DATE
State ORG
Dec 2001   to   May 2004 DATE
1999 DATE
Dec 2001 DATE
State ORG
Education      N/A  ,    ORG
Business Administration ORG
1999 DATE
Jefferson College ORG
State       Business Administration  Marketing / Advertising         High School ORG
College Prep PERSON
1998 DATE
State       Awarded American Shrubel Leadership Scholarship ORG
Jefferson College         Skills     Accounting ORG
Custo

In [66]:
import spacy

print(spacy.__version__)

3.8.14


In [67]:
import spacy

nlp = spacy.load("en_core_web_sm")

In [68]:
text = "Python SQL Machine Learning"

doc = nlp(text)

for token in doc:
    print(token.text)

Python
SQL
Machine
Learning


In [69]:
for token in doc:
    print(token.text, "->", token.lemma_)

Python -> Python
SQL -> SQL
Machine -> Machine
Learning -> Learning


In [70]:
for token in doc:
    print(token.text, token.pos_)

Python PROPN
SQL PROPN
Machine PROPN
Learning PROPN


In [71]:
text = "Arpita worked at Microsoft in Delhi in 2025."

doc = nlp(text)

for ent in doc.ents:
    print(ent.text, ent.label_)

Arpita ORG
Microsoft ORG
Delhi GPE
2025 DATE


In [72]:
resume = df["Resume_str"][0]

doc = nlp(resume)

for token in doc[:50]:
    print(token.text)

         
HR
ADMINISTRATOR
/
MARKETING
ASSOCIATE



HR
ADMINISTRATOR
      
Summary
    
Dedicated
Customer
Service
Manager
with
15
+
years
of
experience
in
Hospitality
and
Customer
Service
Management
.
  
Respected
builder
and
leader
of
customer
-
focused
teams
;
strives
to
instill
a
shared
,
enthusiastic
commitment
to
customer


In [73]:
for ent in doc.ents:
    print(ent.text, ent.label_)

15+ years DATE
Hospitality GPE
Team ORG
Missouri GPE
DOT ORG
IHG ORG
Customer Loyalty and Marketing ORG
Hilton GPE
Training Certification PERSON
Micros NORP
Fidelio     PERSON
Dec 2013 DATE
Current      Company Name ORG
State     Helps ORG
Prepares PERSON
Keeps ORG
Assisted PERSON
2 months DATE
2012 DATE
Dec 2013 DATE
CPT ORG
2010 DATE
Dec 2010 DATE
Established PERSON
daily DATE
Marketing and Advertising ORG
Chamber of Commerce ORG
2007 DATE
Jun 2010      Company Name   －    PERSON
- Executive ORG
Marketing, Customer Service ORG
second ORDINAL
Reservation & Front Office ORG
2004 DATE
2007 DATE
State ORG
Dec 2001   to   May 2004 DATE
1999 DATE
Dec 2001 DATE
State ORG
Education      N/A  ,    ORG
Business Administration ORG
1999 DATE
Jefferson College ORG
State       Business Administration  Marketing / Advertising         High School ORG
College Prep PERSON
1998 DATE
State       Awarded American Shrubel Leadership Scholarship ORG
Jefferson College         Skills     Accounting ORG
Custo

In [75]:
import os

print(os.getcwd())

e:\DATA_SCIENCE\ResumeIQ\notebooks


In [76]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [77]:
print(sys.path[-1])

e:\DATA_SCIENCE\ResumeIQ


In [78]:
from models.skill_extractor import load_skills

In [80]:
resume = df["Clean_Resume"][0]

In [81]:
resume = resume.lower()

In [82]:
found_skills = []

In [84]:
print(found_skills)

[]


In [85]:
print("Total Skills Found :", len(found_skills))

Total Skills Found : 0


In [87]:
print(resume[:1000])

         hr administrator marketing associate

hr administrator       summary     dedicated customer service manager with     years of experience in hospitality and customer service management    respected builder and leader of customer focused teams  strives to instill a shared  enthusiastic commitment to customer service          highlights         focused on customer satisfaction  team management  marketing savvy  conflict resolution techniques     training and development  skilled multi tasker  client relations specialist           accomplishments      missouri dot supervisor training certification  certified by ihg in customer loyalty and marketing by segment   hilton worldwide general manager training certification  accomplished trainer for cross server hospitality systems such as    hilton onq      micros    opera pms     fidelio    opera    reservation system  ors      holidex    completed courses and seminars in customer service  sales strategies  inventory control  loss preve

In [88]:
df["Category"].value_counts()

Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
FINANCE                   118
ENGINEERING               118
ACCOUNTANT                118
FITNESS                   117
AVIATION                  117
SALES                     116
HEALTHCARE                115
CONSULTANT                115
BANKING                   115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64

In [89]:
X = df["Clean_Resume"]

y = df["Category"]

In [90]:
print(X.shape)

print(y.shape)

(2484,)
(2484,)


In [91]:
print(X.iloc[0])

         hr administrator marketing associate

hr administrator       summary     dedicated customer service manager with     years of experience in hospitality and customer service management    respected builder and leader of customer focused teams  strives to instill a shared  enthusiastic commitment to customer service          highlights         focused on customer satisfaction  team management  marketing savvy  conflict resolution techniques     training and development  skilled multi tasker  client relations specialist           accomplishments      missouri dot supervisor training certification  certified by ihg in customer loyalty and marketing by segment   hilton worldwide general manager training certification  accomplished trainer for cross server hospitality systems such as    hilton onq      micros    opera pms     fidelio    opera    reservation system  ors      holidex    completed courses and seminars in customer service  sales strategies  inventory control  loss preve

In [92]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

print(y_encoded[:10])

[19 19 19 19 19 19 19 19 19 19]


In [93]:
print(encoder.classes_)

['ACCOUNTANT' 'ADVOCATE' 'AGRICULTURE' 'APPAREL' 'ARTS' 'AUTOMOBILE'
 'AVIATION' 'BANKING' 'BPO' 'BUSINESS-DEVELOPMENT' 'CHEF' 'CONSTRUCTION'
 'CONSULTANT' 'DESIGNER' 'DIGITAL-MEDIA' 'ENGINEERING' 'FINANCE' 'FITNESS'
 'HEALTHCARE' 'HR' 'INFORMATION-TECHNOLOGY' 'PUBLIC-RELATIONS' 'SALES'
 'TEACHER']


In [94]:
for i, category in enumerate(encoder.classes_):
    print(i, "->", category)

0 -> ACCOUNTANT
1 -> ADVOCATE
2 -> AGRICULTURE
3 -> APPAREL
4 -> ARTS
5 -> AUTOMOBILE
6 -> AVIATION
7 -> BANKING
8 -> BPO
9 -> BUSINESS-DEVELOPMENT
10 -> CHEF
11 -> CONSTRUCTION
12 -> CONSULTANT
13 -> DESIGNER
14 -> DIGITAL-MEDIA
15 -> ENGINEERING
16 -> FINANCE
17 -> FITNESS
18 -> HEALTHCARE
19 -> HR
20 -> INFORMATION-TECHNOLOGY
21 -> PUBLIC-RELATIONS
22 -> SALES
23 -> TEACHER


In [95]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

X_tfidf = tfidf.fit_transform(X)

In [96]:
print(X_tfidf.shape)

(2484, 5000)


In [97]:
print(type(X_tfidf))

<class 'scipy.sparse._csr.csr_matrix'>


In [98]:
print(X_tfidf)

  (0, 2188)	0.1407091921165898
  (0, 110)	0.14205555642137682
  (0, 2745)	0.287916003236355
  (0, 357)	0.0691597050141759
  (0, 4433)	0.013685874258648078
  (0, 1221)	0.03260836144203433
  (0, 1171)	0.2021773110645327
  (0, 4110)	0.13709787350767674
  (0, 2715)	0.06616078297395288
  (0, 4987)	0.018602017990738375
  (0, 1718)	0.022313859431618774
  (0, 2169)	0.09337814621194047
  (0, 2714)	0.07514755092329416
  (0, 3860)	0.05721411735023218
  (0, 597)	0.05020757756789136
  (0, 2551)	0.027892049246852457
  (0, 1869)	0.05838392406055799
  (0, 4523)	0.02632232464047534
  (0, 4132)	0.04442155057822982
  (0, 1606)	0.039561233462376784
  (0, 880)	0.044618139905629656
  (0, 2136)	0.019742181731925816
  (0, 3994)	0.027976850584607336
  (0, 4522)	0.013805724743105006
  (0, 4002)	0.046218907068897294
  :	:
  (2483, 1937)	0.04678862812524783
  (2483, 3216)	0.1098027577229666
  (2483, 4384)	0.08950130751114897
  (2483, 1366)	0.045485024387058974
  (2483, 432)	0.07805620856109899
  (2483, 2200)	0.05

In [99]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y_encoded,
    test_size=0.2,
    random_state=42
)

In [100]:
print("Training Data :", X_train.shape)
print("Testing Data :", X_test.shape)

print("Training Labels :", y_train.shape)
print("Testing Labels :", y_test.shape)

Training Data : (1987, 5000)
Testing Data : (497, 5000)
Training Labels : (1987,)
Testing Labels : (497,)


In [101]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


LogisticRegression(max_iter=1000)

In [102]:
y_pred = model.predict(X_test)

In [103]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy :", accuracy)

Accuracy : 0.6458752515090543


In [104]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy :", accuracy)

Accuracy : 0.6458752515090543


In [105]:
print(X.iloc[0])

         hr administrator marketing associate

hr administrator       summary     dedicated customer service manager with     years of experience in hospitality and customer service management    respected builder and leader of customer focused teams  strives to instill a shared  enthusiastic commitment to customer service          highlights         focused on customer satisfaction  team management  marketing savvy  conflict resolution techniques     training and development  skilled multi tasker  client relations specialist           accomplishments      missouri dot supervisor training certification  certified by ihg in customer loyalty and marketing by segment   hilton worldwide general manager training certification  accomplished trainer for cross server hospitality systems such as    hilton onq      micros    opera pms     fidelio    opera    reservation system  ors      holidex    completed courses and seminars in customer service  sales strategies  inventory control  loss preve

In [106]:
print(X_tfidf.shape)

(2484, 5000)


In [107]:
import joblib

In [108]:
import joblib

joblib.dump(model, "../models/resume_classifier.pkl")

print("Model Saved Successfully")

Model Saved Successfully


In [109]:
joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")

print("Vectorizer Saved Successfully")

Vectorizer Saved Successfully


In [110]:
import joblib

joblib.dump(model, "../models/resume_classifier.pkl")
joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")

print("All files saved successfully!")

All files saved successfully!


In [111]:
resume = df["Resume_str"][0]

print(resume[:1000])

         HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System (ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inventory control, loss preve

In [112]:
resume = df["Resume_str"][0]

print("Skills :", "skills" in resume.lower())
print("Projects :", "projects" in resume.lower())
print("Education :", "education" in resume.lower())
print("Experience :", "experience" in resume.lower())
print("Certifications :", "certification" in resume.lower())

Skills : True
Projects : False
Education : True
Experience : True
Certifications : True


In [113]:
resume = df["Resume_str"][0].lower()

print("summary" in resume)
print("skills" in resume)
print("education" in resume)
print("projects" in resume)

True
True
True
False


In [114]:
from utils.ats_score import calculate_ats_score

resume = df["Resume_str"][0]

score = calculate_ats_score(resume)

print("ATS Score :", score)

ATS Score : (0, {'Email': '❌ Missing', 'Phone': '❌ Missing', 'LinkedIn': '❌ Missing', 'GitHub': '❌ Missing'})


In [115]:
import importlib
import utils.ats_score

importlib.reload(utils.ats_score)

<module 'utils.ats_score' from 'e:\\DATA_SCIENCE\\ResumeIQ\\utils\\ats_score.py'>

In [116]:
from utils.ats_score import calculate_ats_score

resume = df["Resume_str"][0]

score, report = calculate_ats_score(resume)

print("ATS Score :", score)

print("\nResume Report")

for key, value in report.items():
    print(f"{key} : {value}")

ATS Score : 0

Resume Report
Email : ❌ Missing
Phone : ❌ Missing
LinkedIn : ❌ Missing
GitHub : ❌ Missing


In [117]:
resume = df["Resume_str"][0]

score, report = calculate_ats_score(resume)

print(score)
print(report)

0
{'Email': '❌ Missing', 'Phone': '❌ Missing', 'LinkedIn': '❌ Missing', 'GitHub': '❌ Missing'}


In [118]:
from utils.structure_analyzer import analyze_structure

resume = df["Resume_str"][0]

report = analyze_structure(resume)

for key, value in report.items():

    print(f"{key} : {value}")

Summary : True
Skills : True
Projects : False
Experience : True
Education : True
Certifications : True


In [119]:
from utils.project_analyzer import analyze_projects

resume = df["Resume_str"][0]

report = analyze_projects(resume)

for key, value in report.items():
    print(f"{key} : {value}")

GitHub : False
Live Demo : False
Tech Stack : []
Action Words : ['designed', 'created', 'trained']


In [120]:
import os

print(os.getcwd())

e:\DATA_SCIENCE\ResumeIQ\notebooks


In [121]:
import os

print(os.path.exists("../skills"))

True


In [122]:
import os

print(os.path.exists("../skills/programming.txt"))

True


In [123]:
from models.skill_extractor import load_skills

skills = load_skills("../skills/programming.txt")

print(skills[:10])

['python', 'java', 'c', 'c++', 'c#', 'javascript', 'typescript', 'go', 'rust', 'php']


In [124]:
import importlib
import models.skill_extractor

importlib.reload(models.skill_extractor)

<module 'models.skill_extractor' from 'e:\\DATA_SCIENCE\\ResumeIQ\\models\\skill_extractor.py'>

In [125]:
from models.skill_extractor import load_skills, extract_skills

In [126]:
import models.skill_extractor

print(dir(models.skill_extractor))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'extract_skills', 'load_skills', 'os', 're']


In [127]:
import importlib
import models.skill_extractor

importlib.reload(models.skill_extractor)

from models.skill_extractor import load_skills, extract_skills

In [128]:
skills = load_skills("../skills/programming.txt")

resume = df["Resume_str"][0]

found = extract_skills(resume, skills)

print(found)

['swift']


In [129]:
resume = df["Resume_str"][0].lower()

index = resume.find("swift")

print(index)
print(resume[index-100:index+100])

4039
esources.  managed front-end operations to ensure friendly and efficient transactions.  ensured the swift resolution of customer issues to preserve customer loyalty while complying with company polici


In [130]:
from utils.section_parser import parse_sections

resume = df["Resume_str"][0]

sections = parse_sections(resume)

for key, value in sections.items():

    print("="*50)
    print(key.upper())
    print("="*50)
    print(value[:300])

SUMMARY
dedicated customer service manager with 15+ years of
SKILLS
as well as computer
EXPERIENCE
in hospitality and customer service management.   respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         highlights         focused on customer satisfaction  team management  marketing savvy  conflict resolution techn
PROJECTS

EDUCATION
n/a  ,   business administration   1999     jefferson college   －   city  ,   state       business administration  marketing / advertising         high school diploma  ,   college prep. studies   1998     sainte genevieve senior high   －   city  ,   state       awarded american shrubel leadership sc
CERTIFICATIONS
certified by ihg in customer loyalty and marketing by segment   hilton worldwide general manager training certification  accomplished trainer for cross server hospitality systems such as    hilton onq  ,   micros    opera pms   , fidelio    opera    reservation sy

In [131]:
from models.skill_extractor import load_skills, extract_skills
from utils.missing_skills import find_missing_skills

resume = df["Resume_str"][0]

# Detect skills from resume
all_skills = load_skills("../skills/data_science.txt")

found = extract_skills(resume, all_skills)

print("Detected Skills:")
print(found)

print()

# Compare with Data Scientist role
missing = find_missing_skills(
    found,
    "../roles/data_scientist.txt"
)

print("Missing Skills:")
print(missing)

Detected Skills:
['data analysis', 'statistics']

Missing Skills:
['python', 'sql', 'numpy', 'pandas', 'matplotlib', 'seaborn', 'scikit-learn', 'machine learning', 'git', 'github', 'docker', 'aws', 'power bi', 'excel', 'opencv', 'tensorflow', 'pytorch']


In [132]:
from utils.weak_words import detect_weak_words

resume = df["Resume_str"][0]

weak = detect_weak_words(resume)

print("Weak Words Found:")
print(weak)

Weak Words Found:
['knowledge']


In [135]:
from utils.resume_analyzer import analyze_resume

resume = df["Resume_str"][0]

analysis = analyze_resume(resume)

In [136]:
for key, value in analysis.items():

    print("="*50)

    print(key)

    print(value)

ATS Score
0
ATS Report
{'Email': '❌ Missing', 'Phone': '❌ Missing', 'LinkedIn': '❌ Missing', 'GitHub': '❌ Missing'}
Detected Skills
['data analysis', 'statistics']
Missing Skills
['python', 'sql', 'numpy', 'pandas', 'matplotlib', 'seaborn', 'scikit-learn', 'machine learning', 'git', 'github', 'docker', 'aws', 'power bi', 'excel', 'opencv', 'tensorflow', 'pytorch']
Weak Words
['knowledge']


In [138]:
analysis = analyze_resume(resume)

for key, value in analysis.items():
    print("=" * 50)
    print(key)
    print(value)

ATS Score
0
ATS Report
{'Email': '❌ Missing', 'Phone': '❌ Missing', 'LinkedIn': '❌ Missing', 'GitHub': '❌ Missing'}
Detected Skills
['data analysis', 'statistics']
Missing Skills
['python', 'sql', 'numpy', 'pandas', 'matplotlib', 'seaborn', 'scikit-learn', 'machine learning', 'git', 'github', 'docker', 'aws', 'power bi', 'excel', 'opencv', 'tensorflow', 'pytorch']
Weak Words
['knowledge']


In [139]:
analysis = analyze_resume(resume)

print(analysis.keys())

dict_keys(['ATS Score', 'ATS Report', 'Detected Skills', 'Missing Skills', 'Weak Words'])


In [140]:
print(analyze_resume.__module__)

utils.resume_analyzer


In [141]:
import utils.resume_analyzer

print(utils.resume_analyzer.__file__)

e:\DATA_SCIENCE\ResumeIQ\utils\resume_analyzer.py


In [143]:
import importlib
import utils.resume_analyzer

importlib.reload(utils.resume_analyzer)

from utils.resume_analyzer import analyze_resume

analysis = analyze_resume(resume)

print(analysis.keys())

dict_keys(['ATS Score', 'ATS Report', 'Detected Skills', 'Missing Skills', 'Weak Words', 'Resume Strength'])


In [144]:
analysis = analyze_resume(resume)

for key, value in analysis.items():
    print("=" * 50)
    print(key)
    print(value)

ATS Score
0
ATS Report
{'Email': '❌ Missing', 'Phone': '❌ Missing', 'LinkedIn': '❌ Missing', 'GitHub': '❌ Missing'}
Detected Skills
['data analysis', 'statistics']
Missing Skills
['python', 'sql', 'numpy', 'pandas', 'matplotlib', 'seaborn', 'scikit-learn', 'machine learning', 'git', 'github', 'docker', 'aws', 'power bi', 'excel', 'opencv', 'tensorflow', 'pytorch']
Weak Words
['knowledge']
Resume Strength
20


In [149]:
import importlib
import utils.resume_analyzer

importlib.reload(utils.resume_analyzer)

from utils.resume_analyzer import analyze_resume

In [150]:
analysis = analyze_resume(resume)

print(analysis.keys())

dict_keys(['ATS Score', 'ATS Report', 'Detected Skills', 'Missing Skills', 'Weak Words', 'Resume Strength', 'Suggestions'])


In [151]:
analysis = analyze_resume(resume)

for key, value in analysis.items():
    print("=" * 50)
    print(key)
    print(value)

ATS Score
0
ATS Report
{'Email': '❌ Missing', 'Phone': '❌ Missing', 'LinkedIn': '❌ Missing', 'GitHub': '❌ Missing'}
Detected Skills
['data analysis', 'statistics']
Missing Skills
['python', 'sql', 'numpy', 'pandas', 'matplotlib', 'seaborn', 'scikit-learn', 'machine learning', 'git', 'github', 'docker', 'aws', 'power bi', 'excel', 'opencv', 'tensorflow', 'pytorch']
Weak Words
['knowledge']
Resume Strength
20
Suggestions
['Add a professional email address.', 'Add your phone number.', 'Add your LinkedIn profile.', 'Add your GitHub profile.', 'Learn or include these important skills: python, sql, numpy, pandas, matplotlib', 'Replace weak words with strong action verbs.', 'Your resume needs significant improvement.']


In [155]:
import importlib
import utils.resume_analyzer

importlib.reload(utils.resume_analyzer)

from utils.resume_analyzer import analyze_resume

analysis = analyze_resume(resume)

print(analysis["Recruiter View"])


{'Recommendation': 'Needs Improvement', 'Strengths': [], 'Weaknesses': ['Very few technical skills detected.', 'Many important industry skills are missing.', 'LinkedIn profile is missing.', 'GitHub profile is missing.']}


In [157]:
from utils.pdf_parser import extract_text_from_pdf

resume = extract_text_from_pdf("../sample_resumes/MyResume.pdf")

print(resume[:2000])

Arpita Jain
Github https://github.com/arpita-jain840
|
LINKEDIN https://www.linkedin.com/in/arpita-jain-922a58321
|
Envelope
arpijain990@gmail.com
|
Phone 8130598334
SUMMARY
B.Tech Computer Science student with a growing interest in Data Science and Machine Learning. Currently
building skills in data analysis, visualization, and machine learning through hands-on projects, hackathons, and
continuous learning. Passionate about solving real-world problems using technology, while also exploring public
speaking, collaboration, and effective communication to create meaningful impact.
PROJECTS
HR Analytics Dashboard | Power BI, Python (Pandas, NumPy)
• Built an interactive HR Analytics Dashboard to analyze workforce trends including employee count, attrition, salary
distribution, job satisfaction, and gender ratio.
• Performed data cleaning and preprocessing using Python (Pandas, NumPy) and created dynamic visualizations in
Power BI.
• Enabled data-driven HR decision making by identifying key

In [158]:
from utils.resume_analyzer import analyze_resume

analysis = analyze_resume(resume)

for key, value in analysis.items():
    print("=" * 60)
    print(key)
    print(value)

ATS Score
20
ATS Report
{'Email': '✅ Found', 'Phone': '✅ Found', 'LinkedIn': '✅ Found', 'GitHub': '✅ Found'}
Detected Skills
['computer vision', 'data analysis', 'machine learning', 'matplotlib', 'nlp', 'numpy', 'opencv', 'pandas', 'scikit-learn', 'seaborn']
Missing Skills
['python', 'sql', 'statistics', 'git', 'github', 'docker', 'aws', 'power bi', 'excel', 'tensorflow', 'pytorch']
Weak Words
[]
Resume Strength
70
Suggestions
['Learn or include these important skills: python, sql, statistics, git, github', 'Your resume is strong.']
Recruiter View
{'Recommendation': 'Maybe', 'Strengths': ['Good technical skill coverage.', 'Strong overall resume.', 'LinkedIn profile available.', 'GitHub profile available.'], 'Weaknesses': ['Many important industry skills are missing.']}


In [159]:
from ai.recruiter_feedback import build_recruiter_prompt

prompt = build_recruiter_prompt(
    resume,
    analysis
)

print(prompt)


You are a Senior Technical Recruiter with more than 15 years of hiring experience.

Analyze the following resume professionally.

Resume Information
------------------

ATS Score:
20

Detected Skills:
computer vision, data analysis, machine learning, matplotlib, nlp, numpy, opencv, pandas, scikit-learn, seaborn

Missing Skills:
python, sql, statistics, git, github, docker, aws, power bi, excel, tensorflow, pytorch

Weak Words:
None

Resume Strength:
70/100


Resume:

Arpita Jain
Github https://github.com/arpita-jain840
|
LINKEDIN https://www.linkedin.com/in/arpita-jain-922a58321
|
Envelope
arpijain990@gmail.com
|
Phone 8130598334
SUMMARY
B.Tech Computer Science student with a growing interest in Data Science and Machine Learning. Currently
building skills in data analysis, visualization, and machine learning through hands-on projects, hackathons, and
continuous learning. Passionate about solving real-world problems using technology, while also exploring public
speaking, collaboration,

In [162]:
import google.generativeai as genai
from dotenv import load_dotenv
import os

load_dotenv()

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

for model in genai.list_models():
    if "generateContent" in model.supported_generation_methods:
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

In [163]:
import google.generativeai as genai
from dotenv import load_dotenv
import os

load_dotenv()

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

model = genai.GenerativeModel("gemini-flash-latest")

response = model.generate_content("Say Hello ResumeIQ!")

print(response.text)

Hello ResumeIQ! 

How can I help you with your resume, cover letter, or career goals today?


In [164]:
from ai.gemini_client import generate_response

answer = generate_response(
    "Introduce yourself as a Senior HR Recruiter."
)

print(answer)

Here are a few ways to introduce yourself as a Senior HR Recruiter, depending on the context and audience:

### Option 1: To a Job Candidate (Warm, Professional & Engaging)
> "Hello [Candidate Name], 
> 
> My name is [Your Name], and I am a Senior HR Recruiter here at [Company Name]. 
> 
> My primary focus is identifying exceptional talent and helping professionals like you find meaningful career opportunities within our organization. I was really impressed by your background in [Candidate's Skill/Field], and I’d love to connect to discuss how your experience aligns with some of the exciting growth happening on our team."

---

### Option 2: To a Hiring Manager / Internal Stakeholder (Strategic & Results-Oriented)
> "Hi [Name], 
> 
> I’m [Your Name], Senior HR Recruiter supporting [Department/Business Unit]. 
> 
> With over [X] years of experience in talent acquisition, my goal is to serve as a strategic partner to your team. I’m here to help you identify top-tier talent, streamline yo

In [165]:
from ai.recruiter_feedback import build_recruiter_prompt
from ai.gemini_client import generate_response

prompt = build_recruiter_prompt(
    resume,
    analysis
)

feedback = generate_response(prompt)

print(feedback)

Here is a professional evaluation of Arpita Jain's resume from a Senior Technical Recruiter's perspective.

---

### 1. Overall Assessment
Arpita is an academically strong (9.0+ CGPA) B.Tech Computer Science student with a clear target trajectory in Data Analysis, Machine Learning, and Software Development. For a student early in her degree program (2024–2028), the project work is practical and shows good initiative. 

However, **there is a critical disconnect between the resume content and how ATS (Applicant Tracking System) software reads it.** Despite having Python, SQL (MySQL), Excel, Power BI, and GitHub listed on her resume, the parser failed to pick them up (resulting in an ATS score of 20). This usually stems from non-standard character encoding, special characters (e.g., `퐾` in place of 'K'), or layout issue formatting.

---

### 2. Candidate Strengths
* **Quantifiable Metrics in Projects:** Excellent use of hard numbers in the Coffee Shop project (*"149K+ transactions"*, *"id

In [166]:
from ai.recruiter_feedback import build_recruiter_prompt
from ai.gemini_client import generate_response

prompt = build_recruiter_prompt(
    resume,
    analysis
)

feedback = generate_response(prompt)

print(feedback)

Here is my professional evaluation of Arpita Jain's resume based on my 15+ years of technical recruiting experience.

---

### 1. Overall Assessment
Arpita is a highly promising **1st-year B.Tech Computer Science student (Graduating 2028)** with a remarkably strong academic track record (CGPA 9.0+) and an impressive early start in Data Analytics and Machine Learning. 

**Critical ATS Disconnect:** While the visual resume contains high-value technical keywords (Python, MySQL, Power BI, Excel, GitHub), the candidate's actual ATS score is low (20/100). This indicates severe parsing errors caused by non-standard formatting, special character encoding issues (e.g., `"698퐾"`), and graphic icons (e.g., `"Envelope"`). Fixing these technical resume flaws is critical for her applications to pass automated recruiter screens.

---

### 2. Candidate Strengths
* **Exceptional Academic Track Record:** Consistent top-tier academic performance (9.0+ CGPA in university and secondary school) signals high

In [167]:
from ai.job_match import build_job_match_prompt
from ai.gemini_client import generate_response

job_description = """
We are looking for a Data Science Intern.

Requirements

- Python
- SQL
- Machine Learning
- Statistics
- Git
- Power BI
"""

prompt = build_job_match_prompt(
    resume,
    analysis,
    job_description
)

response = generate_response(prompt)

print(response)

### 1. Overall Match Score
**85%**

---

### 2. Overall Assessment
Arpita Jain is a strong candidate for the **Data Science Intern** position. Despite being early in her undergraduate B.Tech program (2024–2028), she demonstrates high academic performance (9.0+ CGPA) and a proactive hands-on approach. Her skills directly map to 5 out of 6 key requirements in the Job Description (Python, SQL, Machine Learning, Git, and Power BI). With a solid baseline of practical projects covering data visualization, ML workflows, and database management, she fits the ideal profile for an entry-level intern.

---

### 3. Matching Skills
* **Python** (NumPy, Pandas, Matplotlib, Seaborn, Scikit-learn)
* **SQL** (MySQL)
* **Machine Learning** (Scikit-learn, NLP/CV concepts)
* **Git** (GitHub, VS Code environment)
* **Power BI** (Interactive Dashboard creation)

---

### 4. Missing Skills
* **Statistics** (Lack of explicit coursework or project demonstration covering inferential statistics, hypothesis testi

In [169]:
from ai.interview_questions import build_interview_prompt
from ai.gemini_client import generate_response

job_description = """
Data Science Intern

Requirements:

Python
SQL
Machine Learning
Statistics
Git
Power BI
"""

prompt = build_interview_prompt(
    resume,
    analysis,
    job_description
)

response = generate_response(prompt)

print(response)

### 1. Overall Difficulty
**Medium**

*Rationale:* As a candidate in the early stages of your B.Tech program, you bring solid foundational skills in Python, basic ML, Power BI, and Excel analysis. However, the job description explicitly emphasizes core fundamentals—specifically **SQL**, **Statistics**, **Git**, and **Machine Learning**. Interviewers will test whether your theoretical knowledge matches the practical claims on your resume, with deep dives into data manipulation, statistical intuition, and query writing.

---

### 2. Technical Questions

1. **Python / Data Structures:** What is the underlying difference in memory storage and computation performance between a standard Python `list` and a `NumPy` array?
2. **Pandas / Data Cleaning:** Explain the functional difference between `.loc` and `.iloc` in Pandas. When handling missing data, how do you decide whether to drop rows (`dropna`) versus imputing values (`fillna`), and how does this impact downstream models?
3. **Statistics

In [170]:
from ai.interview_questions import build_interview_prompt
from ai.gemini_client import generate_response

job_description = """
Data Science Intern

Requirements:

Python
SQL
Machine Learning
Statistics
Git
Power BI
"""

prompt = build_interview_prompt(
    resume,
    analysis,
    job_description
)

response = generate_response(prompt)

print(response)

### 1. Overall Difficulty
**Medium**  
*(Appropriate for an entry-level Data Science Intern role targeting fundamental Python, SQL, foundational Machine Learning concepts, basic Inferential Statistics, and practical project walk-throughs).*

---

### 2. Technical Questions
1. **Python / Pandas:** What is the difference between `loc` and `iloc` in Pandas? How do you handle missing values (`NaN`) in a dataset without dropping entire rows?
2. **Statistics:** Explain the difference between Mean, Median, and Mode. In what scenarios (e.g., skewed distributions like salary data) would you prefer Median over Mean?
3. **Statistics:** What is the Central Limit Theorem (CLT), and why is it crucial for hypothesis testing and inferential statistics in data science?
4. **Machine Learning:** Explain the Bias-Variance Tradeoff. How do underfitting and overfitting manifest in a model, and what techniques can be used to prevent them?
5. **Machine Learning:** What is the difference between Precision, Rec

In [171]:
from ai.resume_rewriter import build_resume_rewriter_prompt
from ai.gemini_client import generate_response

prompt = build_resume_rewriter_prompt(
    resume,
    analysis
)

response = generate_response(prompt)

print(response)

**ARPITA JAIN**  
Email: arpijain990@gmail.com | Phone: +91 8130598334  
LinkedIn: linkedin.com/in/arpita-jain-922a58321 | GitHub: github.com/arpita-jain840  

---

### **PROFESSIONAL SUMMARY**
High-performing Computer Science undergraduate (CGPA: 9.0+) specializing in Data Science, Machine Learning, and Data Analytics. Proficient in Python, SQL, Power BI, and Advanced Excel, with practical experience building data-driven dashboards, machine learning pipelines, and analytics solutions. Skilled at converting complex raw data into actionable strategic insights and communicating technical findings effectively to drive business decisions.

---

### **TECHNICAL SKILLS**
* **Languages:** Python, Java, C++, SQL
* **Data Analysis & Visualization:** Pandas, NumPy, Matplotlib, Seaborn, Power BI, Advanced Microsoft Excel (Pivot Tables, Charts, Dashboards)
* **Machine Learning & AI:** Scikit-Learn, Natural Language Processing (NLP), Computer Vision (OpenCV)
* **Databases:** MySQL, MongoDB
* **Deve

In [172]:
from ai.resume_rewriter import build_resume_rewriter_prompt
from ai.gemini_client import generate_response

prompt = build_resume_rewriter_prompt(
    resume,
    analysis
)

response = generate_response(prompt)

print(response)

**ARPITA JAIN**  
Delhi, India | +91 8130598334 | arpijain990@gmail.com  
[LinkedIn Profile](https://www.linkedin.com/in/arpita-jain-922a58321) | [GitHub Profile](https://github.com/arpita-jain840)

---

### **PROFESSIONAL SUMMARY**
Results-driven Computer Science undergraduate specializing in Data Science, Machine Learning, and Data Analytics. Proficient in Python, SQL, Data Visualization, and predictive modeling, with practical experience building data-driven solutions, interactive dashboards, and machine learning workflows. Adept at extracting actionable insights from complex datasets to solve real-world problems and support strategic decision-making.

---

### **TECHNICAL SKILLS**
* **Programming Languages:** Python, Java, C++
* **Data Analytics & Visualization:** Power BI, Microsoft Excel (Advanced), Pandas, NumPy, Matplotlib, Seaborn
* **Machine Learning & AI:** Scikit-learn, Natural Language Processing (NLP), Computer Vision (OpenCV)
* **Databases:** MySQL, MongoDB
* **Developer

In [173]:
from ai.cover_letter import build_cover_letter_prompt
from ai.gemini_client import generate_response

job_description = """
Data Science Intern

Requirements:

Python
SQL
Machine Learning
Git
Power BI
"""

prompt = build_cover_letter_prompt(
    resume,
    analysis,
    job_description
)

response = generate_response(prompt)

print(response)

Arpita Jain
Phone: 8130598334 | Email: arpijain990@gmail.com
LinkedIn: https://www.linkedin.com/in/arpita-jain-922a58321 | GitHub: https://github.com/arpita-jain840

Dear Hiring Manager,

I am writing to express my strong interest in the Data Science Intern position. As a Computer Science undergraduate at Guru Gobind Singh Indraprastha University (maintaining a 9.0+ CGPA) with a solid foundation in data analysis, machine learning, and visualization, I am excited about the opportunity to contribute to your team and solve real-world problems through data-driven insights.

My technical background aligns directly with the core requirements of the Data Science Intern role. I have practical experience programming in Python and utilizing core data science libraries, including Pandas, NumPy, Matplotlib, Seaborn, and Scikit-learn, to clean, analyze, and build predictive machine learning models. Complementing my Python skills, I have foundational knowledge in database management using SQL (MySQL